# 01. Raw dataset: виртуализация, сэмплирование и проверка chunk cache

В этом ноутбуке мы разберём практический сценарий работы с **одним raw датасетом**:

- выбор датасета по YAML-имени;
- инициализация через проектный API;
- получение батча PIL-изображений;
- проверка дискового chunk-кэша и политики eviction;
- различия `streaming=True/False`.


In [ ]:
# Базовые параметры ноутбука (можно менять)
DATASET_YAML_NAME = "flickr30k"   # пример не-gated датасета
DATA_ROOT = "./data_tutorials"
N_IMAGES = 12
DEMO_NUM_CHUNKS_KEPT = 1
DEMO_CHUNK_SIZE = 8

print("Параметры:")
print("DATASET_YAML_NAME:", DATASET_YAML_NAME)
print("DATA_ROOT:", DATA_ROOT)
print("N_IMAGES:", N_IMAGES)


In [ ]:
from __future__ import annotations

import os
import sys
import platform
import time
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import torch
from PIL import Image
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.data_raw.providers.hf import register_all_adapters
from dataset.data_raw.providers.hf.auth import get_hf_token
from dataset.data_raw.registry import create_dataset
from dataset.models.registry import create_model
from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset
from dataset.shared.compatibility_index import CompatibilityIndex


def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        cfg = compose(config_name=cfg_path.stem, overrides=overrides or [])
    return cfg


def resolve_dataset_cfg(cfg, dataset_yaml_name: str):
    data_cfg = cfg.data
    config_dirs = list(data_cfg.get("dataset_config_dirs", ["conf/data/datasets"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{dataset_yaml_name}.yaml"
        if candidate.exists():
            ds_cfg = OmegaConf.load(candidate)
            override_map = cfg.data.get("dataset_overrides", {})
            if dataset_yaml_name in override_map:
                ds_cfg = OmegaConf.merge(ds_cfg, override_map[dataset_yaml_name])
            return ds_cfg
    raise FileNotFoundError(f"Не найден YAML датасета: {dataset_yaml_name}")


def resolve_model_cfg(cfg, model_yaml_name: str):
    models_cfg = cfg.models if "models" in cfg else {}
    config_dirs = list(models_cfg.get("model_config_dirs", ["conf/data/models"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{model_yaml_name}.yaml"
        if candidate.exists():
            return OmegaConf.load(candidate)
    raise FileNotFoundError(f"Не найден YAML модели: {model_yaml_name}")


def show_image_grid(pil_list, n: int = 16, title: str | None = None):
    if not pil_list:
        print("Список изображений пуст")
        return
    images = pil_list[:n]
    cols = min(4, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    idx = 0
    for r in range(rows):
        for c in range(cols):
            ax = axes[r][c]
            ax.axis("off")
            if idx < len(images):
                ax.imshow(images[idx])
                ax.set_title(f"#{idx}")
            idx += 1
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def print_yaml_section(path_to_yaml: str):
    path = repo_root / path_to_yaml
    if not path.exists():
        print(f"Файл не найден: {path}")
        return
    print("=" * 80)
    print(f"RAW YAML: {path}")
    print("=" * 80)
    print(path.read_text(encoding="utf-8"))
    print("=" * 80)
    print("OmegaConf (resolve=False)")
    print("=" * 80)
    obj = OmegaConf.load(path)
    print(OmegaConf.to_yaml(obj, resolve=False))
    print("=" * 80)
    print("OmegaConf (resolve=True, если возможно)")
    print("=" * 80)
    try:
        print(OmegaConf.to_yaml(obj, resolve=True))
    except Exception as exc:
        print(f"Не удалось resolve=True: {exc}")


def ensure_hf_token(cfg):
    token = get_hf_token(cfg)
    if not token:
        raise ValueError(
            "HF token missing in top-level config: set hf.token. "
            "Для gated ресурсов также примите лицензию на странице HF."
        )
    return token


def print_system_info():
    print(f"Python: {platform.python_version()}")
    print(f"Torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")


In [ ]:
print_system_info()
print("repo_root:", repo_root)

print("
Важно:")
print("- для gated датасетов обязателен cfg.hf.token")
print("- нужно принять лицензию/условия на странице датасета в Hugging Face")


## Quick start (опционально одной ячейкой)

Поставьте `QUICK_START = True`, если хотите быстро проверить весь базовый flow без запуска всех секций вручную.


In [ ]:
QUICK_START = False

if QUICK_START:
    cfg_qs = load_hydra_cfg(overrides=[
        f"data.path={DATA_ROOT}",
        f"data.enabled_datasets=[{DATASET_YAML_NAME}]",
    ])
    register_all_adapters()
    ds_cfg_qs = resolve_dataset_cfg(cfg_qs, DATASET_YAML_NAME)
    ds_qs = create_dataset(
        name=DATASET_YAML_NAME,
        cfg=OmegaConf.to_container(ds_cfg_qs, resolve=True),
        global_root=str(cfg_qs.data.path),
        seed=int(cfg_qs.data.seed),
        hf_cfg=OmegaConf.to_container(cfg_qs.hf, resolve=True),
    )
    ds_qs.start()
    qs_samples = ds_qs.get_batch(min(8, N_IMAGES))
    show_image_grid([s.image for s in qs_samples], n=min(8, len(qs_samples)), title="Quick start")
    print("stats:", ds_qs.stats())
    ds_qs.close()


## Схема YAML для raw-датасета

Ниже показываем **шаблон** `conf/data/datasets/_data_raw_.yaml` и конкретный пример датасета.

| Ключ | Что означает | Почему важно |
|---|---|---|
| `name` | логическое имя датасета | используется в `data.enabled_datasets` |
| `gated` | gated-датасет или нет | при `true` обязателен `hf.token` |
| `hf.repo/subset/split` | источник и выборка в HF | определяет что именно читаем |
| `hf.streaming` | streaming режим | влияет на стратегию выборки и shuffle |
| `schema.image_mode` | `image_field` или `url_field` | как извлекается изображение |
| `cache.chunk_size_images` | размер чанка | влияет на latency/IO |
| `cache.num_chunks_kept` | сколько чанков держать на диске | контроль диска |
| `sampling_weight` | вес датасета в смешивании | вероятность попадания в mixed batch |
| `models` | список совместимых моделей | строит `model -> datasets` индекс |
| `collector_device` | optional override устройства | участвует в async-совместимости |


In [ ]:
print_yaml_section("conf/data/datasets/_data_raw_.yaml")
print_yaml_section(f"conf/data/datasets/{DATASET_YAML_NAME}.yaml")


## Выбор датасета по YAML имени

Выбираем `DATASET_YAML_NAME`, подгружаем top-level конфиг, применяем overrides и создаём объект датасета.


In [ ]:
overrides = [
    f"data.path={DATA_ROOT}",
    f"data.enabled_datasets=[{DATASET_YAML_NAME}]",
]
cfg = load_hydra_cfg(overrides=overrides)

register_all_adapters()

dataset_cfg = resolve_dataset_cfg(cfg, DATASET_YAML_NAME)
print("Resolved dataset cfg:")
print(OmegaConf.to_yaml(dataset_cfg, resolve=False))

if bool(dataset_cfg.get("gated", False)):
    ensure_hf_token(cfg)

dataset = create_dataset(
    name=DATASET_YAML_NAME,
    cfg=OmegaConf.to_container(dataset_cfg, resolve=True),
    global_root=str(cfg.data.path),
    seed=int(cfg.data.seed),
    hf_cfg=OmegaConf.to_container(cfg.hf, resolve=True),
)
dataset.start()
print("dataset.stats():")
print(dataset.stats())


## Быстрый старт: сэмплирование изображений


In [ ]:
samples = dataset.get_batch(N_IMAGES)
pil_list = [item.image for item in samples]

show_image_grid(pil_list, n=min(N_IMAGES, 16), title=f"{DATASET_YAML_NAME}: sampled images")

print("Первые source_id:", [item.sample_id for item in samples[:10]])
print("dataset_name в батче:", sorted(set(item.dataset_name for item in samples)))


## Проверка chunk cache на диске


In [ ]:
def inspect_chunk_cache(dataset_root: Path):
    chunks_root = dataset_root / "chunks"
    print("dataset_root:", dataset_root)
    print("chunks_root:", chunks_root)
    if not chunks_root.exists():
        print("chunks_root ещё не создан")
        return

    chunk_dirs = sorted([p for p in chunks_root.iterdir() if p.is_dir() and p.name.startswith("chunk_")])
    print(f"Всего chunk dirs: {len(chunk_dirs)}")
    for chunk_dir in chunk_dirs:
        num_images = len(list(chunk_dir.glob("*.jpg"))) + len(list(chunk_dir.glob("*.png")))
        print(f"  - {chunk_dir.name}: images={num_images}")

inspect_chunk_cache(Path(cfg.data.path) / DATASET_YAML_NAME)


In [ ]:
# Несколько дополнительных сэмплирований, чтобы увидеть рост/смену чанков
for i in range(5):
    _ = dataset.get_batch(N_IMAGES)
    print(f"iter={i} stats=", dataset.stats())

inspect_chunk_cache(Path(cfg.data.path) / DATASET_YAML_NAME)


### Демонстрация eviction (ограничиваем число чанков)

Создадим второй инстанс с маленькими значениями кэша и убедимся, что на диске остаётся только `N` чанков.


In [ ]:
demo_cfg = OmegaConf.create(OmegaConf.to_container(dataset_cfg, resolve=True))
demo_cfg.cache.num_chunks_kept = DEMO_NUM_CHUNKS_KEPT
demo_cfg.cache.chunk_size_images = DEMO_CHUNK_SIZE

demo_dataset_name = f"{DATASET_YAML_NAME}_eviction_demo"
demo_cfg.name = demo_dataset_name

demo_dataset = create_dataset(
    name=DATASET_YAML_NAME,
    cfg=OmegaConf.to_container(demo_cfg, resolve=True),
    global_root=str(cfg.data.path),
    seed=int(cfg.data.seed),
    hf_cfg=OmegaConf.to_container(cfg.hf, resolve=True),
)
demo_dataset.start()

for i in range(10):
    _ = demo_dataset.get_batch(DEMO_CHUNK_SIZE)

demo_root = Path(cfg.data.path) / demo_dataset_name
inspect_chunk_cache(demo_root)
print("Ожидаем, что чанков не больше:", DEMO_NUM_CHUNKS_KEPT)


## Streaming vs non-streaming

Если датасет поддерживает `streaming`, можно включить его override-ом и сравнить поведение.


In [ ]:
streaming_cfg = OmegaConf.create(OmegaConf.to_container(dataset_cfg, resolve=True))
streaming_cfg.hf.streaming = True
streaming_cfg.hf.shuffle_buffer = 2000

print("Streaming override:")
print(OmegaConf.to_yaml(streaming_cfg.hf, resolve=True))

print("Пояснение:")
print("- streaming=True: данные читаются итератором, нет полноценного random index")
print("- shuffle_buffer: размер окна случайного перемешивания")


## Troubleshooting

- **Ошибка токена**: `HF token missing in top-level config: set hf.token`.
  - Проверьте `conf/config.yaml -> hf.token`.
- **403 / gated dataset**:
  - Примите лицензию на странице датасета HF, проверьте права токена.
- **trust_remote_code**:
  - Включайте только при необходимости и только для доверенных репозиториев.
- **Недостаток диска**:
  - Уменьшите `chunk_size_images` и `num_chunks_kept`.
- **Проблемы декодирования URL-изображений**:
  - Это нормально для части ссылок; смотрите retry/skip поведение в логах.


In [ ]:
# Cleanup
try:
    dataset.close()
except Exception:
    pass

try:
    demo_dataset.close()
except Exception:
    pass

print("Готово")
